# TCGA-BRCA Treatment–OS Overlap V1 Review

This notebook reviews the saved TCGA-BRCA treatment–OS overlap audit v1 outputs from disk only.
It does not re-run the audit script, re-read raw biotab files, aggregate treatment records,
normalize drug names, or perform modeling of any kind.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'treatment-prep'
    / 'tcga_brca_treatment_os_overlap_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest treatment–OS overlap v1 pointer not found: {latest_pointer_path}. '
        f'Run script 19 first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))

overlap_summary_path = repo_root / latest_pointer['treatment_os_overlap_summary_tsv']
overlap_counts_path = repo_root / latest_pointer['treatment_os_overlap_counts_tsv']
distribution_path = repo_root / latest_pointer['drug_therapy_type_distribution_tsv']
patient_summary_path = repo_root / latest_pointer['drug_therapy_type_patient_summary_tsv']
audit_summary_path = repo_root / latest_pointer['treatment_os_overlap_audit_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']

for required_path in [
    overlap_summary_path, overlap_counts_path, distribution_path,
    patient_summary_path, audit_summary_path, run_log_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required artifact not found: {required_path}')

overlap_summary_df = read_tsv(overlap_summary_path)
overlap_counts_df = read_tsv(overlap_counts_path)
distribution_df = read_tsv(distribution_path)
patient_summary_df = read_tsv(patient_summary_path)
audit_summary_df = read_tsv(audit_summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)

print(f"Run ID    : {latest_pointer['treatment_os_overlap_v1_run_id']}")
print(f"OS ep ID  : {latest_pointer['os_endpoint_v1_run_id']}")
print(f"Cohort ID : {latest_pointer['cohort_v1_build_id']}")
print(f"Pointer   : {latest_pointer_path}")

In [ ]:
# --- Write review tables to 05-results/ ---

os107_path = results_root / '107_treatment_os_overlap_summary.tsv'
os108_path = results_root / '108_treatment_os_overlap_counts.tsv'
os109_path = results_root / '109_drug_therapy_type_distribution.tsv'
os110_path = results_root / '110_drug_therapy_type_patient_summary.tsv'
os111_path = results_root / '111_treatment_os_overlap_audit_summary.tsv'

overlap_summary_df.to_csv(os107_path, sep='\t', index=False)
overlap_counts_df.to_csv(os108_path, sep='\t', index=False)
distribution_df.to_csv(os109_path, sep='\t', index=False)
patient_summary_df.to_csv(os110_path, sep='\t', index=False)
audit_summary_df.to_csv(os111_path, sep='\t', index=False)

print(f'Saved: {os107_path}')
print(f'Saved: {os108_path}')
print(f'Saved: {os109_path}')
print(f'Saved: {os110_path}')
print(f'Saved: {os111_path}')

In [ ]:
# --- Pointer and validation summary ---

print('=== Latest pointer ===')
display(pd.DataFrame([latest_pointer]))

validation = run_log.get('validation', {})
print('\n=== Validation ===')
display(
    pd.DataFrame([
        {'check': k, 'value': str(v)}
        for k, v in validation.items()
    ])
)

print('\n=== Key counts ===')
counts = run_log.get('counts', {})
display(
    pd.DataFrame([
        {'metric': k, 'value': str(v)}
        for k, v in counts.items()
    ])
)

In [ ]:
# --- Drug / radiation overlap breakdown ---

# Drug overlap by os_event
drug_event_df = (
    overlap_summary_df
    .groupby(['has_drug_row', 'os_event'], as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['has_drug_row', 'os_event'], ascending=[False, True])
    .reset_index(drop=True)
)

# Radiation overlap by os_event
rad_event_df = (
    overlap_summary_df
    .groupby(['has_radiation_row', 'os_event'], as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['has_radiation_row', 'os_event'], ascending=[False, True])
    .reset_index(drop=True)
)

# drug_row_count distribution
drug_count_dist_df = (
    overlap_summary_df['drug_row_count']
    .astype(int)
    .value_counts()
    .rename_axis('drug_row_count')
    .reset_index(name='patient_count')
    .sort_values('drug_row_count')
    .reset_index(drop=True)
)

print('=== Drug overlap by os_event ===')
display(drug_event_df)

print('\n=== Radiation overlap by os_event ===')
display(rad_event_df)

print('\n=== Drug row count distribution (patients) ===')
display(drug_count_dist_df)

In [ ]:
# --- Therapy type structure ---

# single_or_mixed distribution
som_df = (
    overlap_summary_df['drug_therapy_type_single_or_mixed']
    .replace('', 'not_applicable')
    .value_counts()
    .rename_axis('drug_therapy_type_single_or_mixed')
    .reset_index(name='patient_count')
    .sort_values('patient_count', ascending=False)
    .reset_index(drop=True)
)

# Dominant therapy type distribution (patients with drug rows only)
dom_df = (
    overlap_summary_df
    .loc[overlap_summary_df['has_drug_row'] == 'yes', 'dominant_therapy_type_if_any']
    .replace('', '[no dominant]')
    .value_counts()
    .rename_axis('dominant_therapy_type')
    .reset_index(name='patient_count')
    .sort_values('patient_count', ascending=False)
    .reset_index(drop=True)
)

print('=== Therapy type single_or_mixed (all 1097 patients) ===')
display(som_df)

print('\n=== Dominant therapy type (patients with drug rows only) ===')
display(dom_df)

print('\n=== Patient-level therapy type summary ===')
display(patient_summary_df)

print('\n=== Drug therapy type distribution (first 20 rows) ===')
display(distribution_df.head(20))

In [ ]:
# --- No-drug-record patients: structural comparison ---

no_drug_df = overlap_summary_df.loc[overlap_summary_df['has_drug_row'] == 'no'].copy()
has_drug_df = overlap_summary_df.loc[overlap_summary_df['has_drug_row'] == 'yes'].copy()

def event_rate(df: pd.DataFrame) -> str:
    total = len(df)
    events = (df['os_event'] == '1').sum()
    return f"{events}/{total} ({100*events/total:.1f}%)"

compare_rows = [
    {'group': 'has_drug_row=yes', 'n': len(has_drug_df), 'os_event_rate': event_rate(has_drug_df)},
    {'group': 'has_drug_row=no',  'n': len(no_drug_df),  'os_event_rate': event_rate(no_drug_df)},
]

# Confounder distributions
for col in ['er_status_by_ihc', 'her2_status_by_ihc', 'ajcc_pathologic_tumor_stage']:
    print(f'\n=== {col} by drug record presence ===')
    ct = (
        overlap_summary_df
        .groupby(['has_drug_row', col], as_index=False)
        .size()
        .rename(columns={'size': 'patient_count'})
        .sort_values(['has_drug_row', 'patient_count'], ascending=[False, False])
        .reset_index(drop=True)
    )
    display(ct)

print('\n=== OS event rate by drug record presence ===')
display(pd.DataFrame(compare_rows))

In [ ]:
# --- Audit summary and feasibility ---

print('=== Treatment–OS overlap audit summary ===')
display(audit_summary_df)

feasibility_row = audit_summary_df.loc[
    audit_summary_df['metric'] == 'feasibility_interpretation'
]
if not feasibility_row.empty:
    feasibility = feasibility_row.iloc[0]['value']
    notes = feasibility_row.iloc[0]['notes']
    print(f'\nFeasibility: {feasibility}')
    print(f'Notes: {notes}')

print('\n=== Full overlap counts ===')
display(overlap_counts_df)

print('\n=== Overlap summary (first 20 rows) ===')
display(overlap_summary_df.head(20))